# Binary Black Hole Waveform Forecasting — Resubmission Notebook (v2)

Rebuilt after the Scientific Reports rejection. Key changes from v1:

| Change | Reason |
|---|---|
| Target is the **noiseless** waveform, not noisy strain | v1's target contained unpredictable noise, capping overlap near 0.45 |
| Simulated IMRPhenomD bank, split by **source parameters** | Editor pt 1, 2; R1 major 1; R2 pt 1 |
| Three single-factor OOD sets (`q`, `spin`, `mass`) | Editor pt 3 — isolates *where* generalization fails |
| Merger time **jittered** within the forecast window | v1 aligned every merger to a fixed index, leaking coalescence time |
| 1024 Hz, 1.0 s context to 0.5 s forecast | signal is bandlimited to 300 Hz; 4096 Hz was 4x oversampled |
| 5 seeds + paired t-test on every architecture claim | Editor pt 4, 8 — n=1 cannot support a ranking |
| Per-run checkpoint directories | v1 overwrote `lstm_best.keras` across runs; evaluated models were not trained models |
| AR-64 / persistence baselines run **before** training | cheapest possible check that the task is non-trivial |

| Section | Description |
|---|---|
| A | Setup |
| B | Download real strain (for noise + real-event evaluation) |
| C | Preprocessing, waveform bank, noise pool, injection |
| D | Loss + metrics |
| E | Architectures + matched-parameter search |
| F | Baselines, callbacks, multi-seed training |
| G | Evaluation |
| H | Ablations |
| I | Final table |
| J | Robustness controls |

**Prerequisite:** `bank_jitter.npz` and `bank_aligned.npz` must exist in `WORK_DIR`.
Generate them with `build_bank.py` under WSL/Linux (`pycbc` has no Windows wheels),
writing to `/mnt/d/GW_Research`.

## SECTION A — Setup

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf

WORK_DIR = os.environ.get('GW_WORK_DIR', r'D:\GW_Research')
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

print(f'Working directory: {os.getcwd()}')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU devices: {gpus}')
print('GPU is READY' if gpus else 'WARNING: No GPU found - fix CUDA before training')

In [ ]:
# ============================================================
# CELL 2: IMPORTS
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import h5py
import gc
import time
import urllib.request
import pickle

from scipy.signal import welch, butter, filtfilt, hilbert
from scipy import signal as scipy_signal
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('=' * 60)
print('All imports successful')
print(f'NumPy:       {np.__version__}')
print(f'TensorFlow:  {tf.__version__}')
print(f'h5py:        {h5py.__version__}')
print('=' * 60)

## SECTION B — Download real strain data

In [ ]:
# ============================================================
# CELL 3: DOWNLOAD ALL GW EVENT FILES (via gwosc package)
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'gwosc', 'requests', '--quiet'])
from gwosc.locate import get_urls
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
session = requests.Session()
retries = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

EVENTS_GPS = {
    'GW150914': 1126259446,
    'GW151226': 1135136350,
    'GW170814': 1186741845,
    'GW170104': 1167559920,   # merger GPS 1167559936.6, centered -16s — OOD TEST, do not add to TRAIN_EVENTS
    'GW170608': 1180922478,   # merger GPS 1180922494.49, centered -16s
    'GW170729': 1185389791,   # merger GPS 1185389807.3, centered -16s
    'GW170823': 1187529240,   # merger GPS 1187529256.5, centered -16s
}

FILES = {}
for name, t0 in EVENTS_GPS.items():
    for det in ['H1', 'L1']:
        urls = get_urls(det, t0, t0 + 32, sample_rate=4096)
        exact = [u for u in urls if f'{t0}-32' in u] or urls
        if not exact:
            print(f'   ⚠️  No URL found for {name} {det}')
            continue
        url = exact[0]
        fname = f'{det}_{name}_{t0}-32.hdf5'
        FILES[fname] = url

print(f'Resolved {len(FILES)} download URLs via GWOSC API.\n')
for fname, url in FILES.items():
    fpath = os.path.join(WORK_DIR, fname)
    if os.path.exists(fpath):
        print(f'   ✅ {fname} already exists, skipping')
        continue
    print(f'   ⬇️  Downloading {fname}')
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        with open(fpath, 'wb') as f:
            f.write(r.content)
        print(f'      saved ({len(r.content)/1e6:.1f} MB)')
    except Exception as e:
        print(f'   ❌ FAILED: {e}')
print('\nDownload step finished.')

In [ ]:
def find_l1_lag_samples(h1_zone, l1_zone, fs=4096, max_lag_ms=15):
    """Cross-correlate H1 and L1 (already whitened/bandpassed) to find the
    sample lag that best aligns them. Search window capped at ±15ms —
    generous relative to LIGO's ~10ms max light-travel-time between
    Hanford and Livingston, so this can't lock onto a spurious peak far
    outside physically possible detector separation."""
    max_lag = int(max_lag_ms * fs / 1000)
    h1_c = h1_zone - np.mean(h1_zone)
    l1_c = l1_zone - np.mean(l1_zone)
    corr = np.correlate(h1_c, l1_c, mode='full')
    lags = np.arange(-len(l1_c) + 1, len(h1_c))
    center = len(corr) // 2
    window = corr[center - max_lag: center + max_lag + 1]
    window_lags = lags[center - max_lag: center + max_lag + 1]
    best_lag = int(window_lags[np.argmax(np.abs(window))])
    print(f"   Cross-correlation lag: {best_lag} samples ({best_lag/fs*1000:.2f} ms)")
    return max(best_lag, 0)  # your load_merger_window only shifts L1 forward, so clip at 0

## SECTION C — Preprocessing, waveform bank, noise pool

In [ ]:
# ============================================================
# CELL 4: PREPROCESSING — off-source PSD whitening + notch filter
# ============================================================
import gc
from scipy.signal.windows import tukey
from scipy.signal import iirnotch

def normalize(data):
    std = np.std(data)
    if std < 1e-10:
        return np.zeros_like(data)
    return (data - np.mean(data)) / std

def estimate_offsource_psd(fname, dt, merger_sample, fs, duration_s=64, offset_s=300, max_tries=5):
    n_dur = int(duration_s * fs)
    with h5py.File(fname, 'r') as f:
        total_len = f['strain']['Strain'].shape[0]
        for attempt in range(max_tries):
            off = offset_s * (attempt + 1)  # push further away each retry: 300s, 600s, 900s...
            for direction in [-1, 1]:
                n_off = int(off * fs)
                start = merger_sample + direction * n_off
                start = max(0, min(start, total_len - n_dur))
                bg = f['strain']['Strain'][start:start + n_dur].astype(np.float64)
                if np.all(np.isfinite(bg)) and np.std(bg) > 1e-25:
                    print(f"      [off-source PSD] clean background at [{start}:{start+n_dur}] "
                          f"({(start - merger_sample)/fs:+.1f}s from merger, attempt {attempt+1})")
                    bg = bg - np.mean(bg)
                    nperseg = min(int(fs * 2), len(bg) // 4)
                    f_ax, psd = welch(bg, fs, nperseg=nperseg)
                    return f_ax, psd
    raise ValueError(f"Could not find a clean off-source background segment for {fname} "
                      f"after {max_tries} attempts")

def whiten_signal_stable(strain, dt, f_ax=None, psd=None):
    fs = 1.0 / dt
    if f_ax is None or psd is None:
        print("      [WARNING] whiten_signal_stable fell back to ON-SOURCE PSD (no bg passed)")
        nperseg = min(int(fs * 2), len(strain) // 4)
        f_ax, psd = welch(strain, fs, nperseg=nperseg)
    rfft_freqs   = np.fft.rfftfreq(len(strain), dt)
    interp_psd   = np.interp(rfft_freqs, f_ax, psd)
    psd_floor    = np.median(interp_psd) * 0.01
    interp_psd   = np.clip(interp_psd, psd_floor, None)
    strain_fft   = np.fft.rfft(strain)
    denom        = np.sqrt(interp_psd / (2.0 / fs) + 1e-30)
    whitened_fft = strain_fft / denom
    out = np.fft.irfft(whitened_fft, n=len(strain))
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def bandpass_filter(data, lowcut=30, highcut=300, fs=4096, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data)

def notch_filter(data, freq, fs, quality=30):
    """Remove a narrow contaminating spectral line without touching
    neighboring frequencies. Added after diagnosing a ~68 Hz narrow-band
    instrumental artifact specific to GW170814 H1/L1 (not present in
    GW150914/GW151226), confirmed via whitened-spectrum comparison across
    events and a single-channel self-fit isolation test. Applied to every
    event for consistency; harmless where no real power sits at this
    frequency."""
    b, a = iirnotch(freq, quality, fs)
    return filtfilt(b, a, data)

def process_strain(raw, dt, label="", max_bad_frac=0.05, f_ax=None, psd=None):
    fs = int(round(1.0 / dt))
    n_bad = np.sum(~np.isfinite(raw))
    bad_frac = n_bad / len(raw)
    if bad_frac > max_bad_frac:
        raise ValueError(f"Too many non-finite samples{' (' + label + ')' if label else ''}")
    elif n_bad > 0:
        idx = np.arange(len(raw)); good = np.isfinite(raw)
        raw = raw.copy(); raw[~good] = np.interp(idx[~good], idx[good], raw[good])
    s = raw - np.mean(raw)
    raw_std = np.std(raw)
    if raw_std < 1e-25:
        raise ValueError(f"Degenerate raw strain segment{' (' + label + ')' if label else ''}")
    s = whiten_signal_stable(s, dt, f_ax=f_ax, psd=psd)
    s = bandpass_filter(s, fs=fs)
    s = notch_filter(s, 68.0, fs=fs)   # remove event-specific ~68 Hz artifact (see notch_filter docstring)
    s = normalize(s)
    return np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def load_merger_window(h1_file, l1_file, gps_file_start, gps_coalescence,
                        l1_lag_samples, window_before=2.0, window_after=0.5, pad=4.0):
    with h5py.File(h1_file, 'r') as f:
        dt = float(f['strain']['Strain'].attrs['Xspacing'])
        total_len = f['strain']['Strain'].shape[0]
    fs = int(round(1.0 / dt))
    merger_sample  = int((gps_coalescence - gps_file_start) * fs)
    before_samples = int(window_before * fs)
    after_samples  = int(window_after * fs)
    pad_samples    = int(pad * fs)
    slice_start = max(0, merger_sample - before_samples - pad_samples)
    slice_end   = min(total_len, merger_sample + after_samples + pad_samples)
    merger_in_slice = merger_sample - slice_start
    zone_start = merger_in_slice - before_samples
    zone_end   = merger_in_slice + after_samples
    print(f"   total_len={total_len:,} | fs={fs}Hz")
    print(f"   merger_sample={merger_sample} | merger_in_slice={merger_in_slice}")
    print(f"   slice=[{slice_start}:{slice_end}] len={slice_end-slice_start}")
    print(f"   zone=[{zone_start}:{zone_end}] len={zone_end-zone_start}")
    if zone_start < 0 or zone_end > (slice_end - slice_start) or zone_end <= zone_start:
        raise ValueError(f"Bad zone: zone_start={zone_start}, zone_end={zone_end}")
    with h5py.File(h1_file, 'r') as f:
        h1_raw = f['strain']['Strain'][slice_start:slice_end].astype(np.float64)
    with h5py.File(l1_file, 'r') as f:
        l1_raw = f['strain']['Strain'][slice_start:slice_end].astype(np.float64)
    f_ax_h1, psd_h1 = estimate_offsource_psd(h1_file, dt, merger_sample, fs)
    f_ax_l1, psd_l1 = estimate_offsource_psd(l1_file, dt, merger_sample, fs)
    h1_proc = process_strain(h1_raw, dt, f_ax=f_ax_h1, psd=psd_h1); del h1_raw; gc.collect()
    l1_proc = process_strain(l1_raw, dt, f_ax=f_ax_l1, psd=psd_l1); del l1_raw; gc.collect()
    h1_zone = h1_proc[zone_start:zone_end].copy()
    l1_zone = l1_proc[zone_start:zone_end].copy()
    del h1_proc, l1_proc; gc.collect()
    if l1_lag_samples > 0 and l1_lag_samples < len(l1_zone):
        l1_zone = np.concatenate([l1_zone[l1_lag_samples:], np.zeros(l1_lag_samples, dtype=np.float32)])
    print(f"   H1: {len(h1_zone)} samples | range=[{h1_zone.min():.3f},{h1_zone.max():.3f}] | NaN:{np.isnan(h1_zone).any()}")
    print(f"   L1: {len(l1_zone)} samples | range=[{l1_zone.min():.3f},{l1_zone.max():.3f}] | NaN:{np.isnan(l1_zone).any()}")
    return h1_zone, l1_zone, dt

print('Preprocessing (off-source PSD + 68 Hz notch) defined.')


### C1 — Configuration and waveform bank

`bank_*.npz` is produced by `build_bank.py` (run under WSL/Linux, `pycbc`).
Set `BANK_MODE` to `'jitter'` for the realistic unknown-coalescence-time setting,
or `'aligned'` for the easier fixed-merger-position setting. Report both.

In [ ]:
from scipy.signal import resample_poly

FS_RAW       = 4096
FS           = 1024
CONTEXT_LEN  = 1024
FORECAST_LEN = 32
WINDOW_MODE  = 'inspiral'
SEG_LEN      = 1536
N_FEATURES   = 2
STRIDE       = 64
PATCH = 8
SNR_RANGE    = (8.0, 40.0)

BANK_MODE = 'jitter'
BANK_FILE = f'bank_{BANK_MODE}.npz'

OOD_SETS = ['ood_q', 'ood_spin', 'ood_mass']

def to_model_rate(x):
    return resample_poly(np.asarray(x, dtype=np.float64), FS, FS_RAW).astype(np.float32)

bank = np.load(BANK_FILE)
PARAM_KEYS = list(bank['param_keys'])
print(f'Loaded {BANK_FILE}')
for k in ['train', 'val'] + OOD_SETS:
    print(f'  bank_{k:9s} {bank[f"bank_{k}"].shape}')
print(f'  param keys: {PARAM_KEYS}')
print(f'  context {CONTEXT_LEN} ({CONTEXT_LEN/FS:.2f}s) -> forecast {FORECAST_LEN} ({FORECAST_LEN/FS:.2f}s) at {FS} Hz')

_pk = np.argmax(np.abs(bank['bank_train'][:, CONTEXT_LEN:]), axis=1)
print(f'  merger index within forecast window: mean {_pk.mean():.0f} std {_pk.std():.0f} '
      f'range [{_pk.min()}, {_pk.max()}]')
if _pk.std() < 1.0:
    print('  NOTE: merger position is fixed -> aligned mode (coalescence time is given to the model)')

### C2 — Noise pool from real detector strain

Whitened, notched H1/L1 segments taken far from any real merger, resampled to
1024 Hz. These supply realistic non-Gaussian noise for injection, which is what
lets the paper claim real detector noise rather than Gaussian simulation.

In [ ]:
NOISE_SOURCE_FILES = [
    ('H1_GW150914_1126259446-32.hdf5', 'L1_GW150914_1126259446-32.hdf5'),
    ('H1_GW170608_1180922478-32.hdf5', 'L1_GW170608_1180922478-32.hdf5'),
    ('H1_GW170729_1185389791-32.hdf5', 'L1_GW170729_1185389791-32.hdf5'),
    ('H1_GW170823_1187529240-32.hdf5', 'L1_GW170823_1187529240-32.hdf5'),
]

def build_noise_pool(n_segments=800, seconds_per_file=20.0, seed=5):
    rng = np.random.RandomState(seed)
    chunks = []
    for h1f, l1f in NOISE_SOURCE_FILES:
        if not (os.path.exists(h1f) and os.path.exists(l1f)):
            print(f'   skipping missing pair {h1f}')
            continue
        pair = []
        for fn in (h1f, l1f):
            with h5py.File(fn, 'r') as f:
                d = f['strain']['Strain']
                dt = float(d.attrs['Xspacing'])
                take = min(d.shape[0], int(seconds_per_file * FS_RAW))
                raw = d[:take].astype(np.float64)
            if np.mean(~np.isfinite(raw)) > 0.05 or np.std(raw[np.isfinite(raw)]) < 1e-25:
                pair = []
                break
            f_ax, psd = welch(raw[np.isfinite(raw)] - np.mean(raw[np.isfinite(raw)]),
                              1.0 / dt, nperseg=int(FS_RAW * 2))
            proc = process_strain(raw, dt, f_ax=f_ax, psd=psd)
            proc = to_model_rate(proc)
            edge = int(2 * FS)
            pair.append(proc[edge:-edge])
        if len(pair) != 2:
            continue
        m = min(len(pair[0]), len(pair[1]))
        chunks.append(np.stack([pair[0][:m], pair[1][:m]], axis=1))
        print(f'   {h1f[3:]}: {m} samples usable')
    if not chunks:
        raise RuntimeError('no usable noise source files')
    pool = []
    while len(pool) < n_segments:
        c = chunks[rng.randint(len(chunks))]
        if len(c) <= SEG_LEN:
            continue
        s = rng.randint(0, len(c) - SEG_LEN)
        seg = c[s:s + SEG_LEN]
        if np.all(np.isfinite(seg)) and seg.std() > 1e-12:
            pool.append(seg / seg.std())
    pool = np.stack(pool).astype(np.float32)
    print(f'Noise pool: {pool.shape}')
    return pool

noise_pool = build_noise_pool()

### C3 — Injection

`X` is noisy strain (signal + real noise) over the context window.
`Y` is the **noiseless** signal over the forecast window. Both are scaled by the
same per-example factor so the model cannot infer the SNR from the target norm.

In [ ]:
def window_bounds(bank_arr, mode, FL):
    n = len(bank_arr)
    if mode == 'inspiral':
        c1 = np.full(n, CONTEXT_LEN)
    else:
        c1 = np.argmax(np.abs(bank_arr), axis=1) - FL + 1
    c0 = c1 - CONTEXT_LEN
    ok = (c0 >= 0) & (c1 + FL <= bank_arr.shape[1])
    return c0, c1, ok

def inject(bank_arr, pool, seed, snr_range=SNR_RANGE, mode=None, FL=None):
    mode = WINDOW_MODE if mode is None else mode
    FL = FORECAST_LEN if FL is None else FL
    rng = np.random.RandomState(seed)
    c0, c1, ok = window_bounds(bank_arr, mode, FL)
    bank_arr, c0, c1 = bank_arr[ok], c0[ok], c1[ok]
    n = len(bank_arr)
    X = np.zeros((n, CONTEXT_LEN, N_FEATURES), np.float32)
    Y = np.zeros((n, FL, N_FEATURES), np.float32)
    snrs = np.zeros(n, np.float32)
    for i in range(n):
        w = bank_arr[i]
        nz = pool[rng.randint(len(pool))]
        snr = rng.uniform(*snr_range)
        for ch in range(N_FEATURES):
            z = nz[:, ch]
            scale = snr / (np.linalg.norm(w) / (np.std(z) + 1e-12) + 1e-12)
            s = w * scale
            seg = s + z[:len(s)]
            X[i, :, ch] = seg[c0[i]:c1[i]]
            Y[i, :, ch] = s[c1[i]:c1[i] + FL]
        snrs[i] = snr
    m = X.std(axis=(1, 2), keepdims=True) + 1e-8
    return (X / m).astype(np.float32), (Y / m).astype(np.float32), snrs

X_train, Y_train, snr_train = inject(bank['bank_train'], noise_pool, 11)
X_val,   Y_val,   snr_val   = inject(bank['bank_val'],   noise_pool, 12)
ood_data = {k: inject(bank[f'bank_{k}'], noise_pool, 13) for k in OOD_SETS}
X_test, Y_test, snr_test = ood_data['ood_q']

print(f'mode={WINDOW_MODE}  context {CONTEXT_LEN/FS:.2f}s -> forecast {FORECAST_LEN/FS*1000:.0f} ms')
print(f'  Train {X_train.shape}  Val {X_val.shape}')
for k in OOD_SETS:
    print(f'  {k:9s} {ood_data[k][0].shape}')

### C4 — Real event definitions

Used only by the robustness controls (Section J) and the real-data evaluation.
No real event is ever trained on under the v2 design.

In [ ]:
def get_event_params(h1_file, l1_file, gps_coalescence, l1_lag, window_before, window_after):
    with h5py.File(h1_file, 'r') as f:
        gps_start = float(f['strain']['Strain'].attrs['Xstart'])
        dt        = float(f['strain']['Strain'].attrs['Xspacing'])
        total_len = int(f['strain']['Strain'].attrs['Npoints'])
    fs = int(round(1.0 / dt))
    print(f"   {os.path.basename(h1_file)}: start {gps_start}, "
          f"merger +{gps_coalescence - gps_start:.3f}s, {total_len/fs:.1f}s long")
    return gps_start

def resolve_event(name, gps, coalescence):
    h1, l1 = f'H1_{name}_{gps}-32.hdf5', f'L1_{name}_{gps}-32.hdf5'
    gps_start = get_event_params(h1, l1, coalescence, 0, 2.0, 0.5)
    h1z, l1z, _ = load_merger_window(h1, l1, gps_start, coalescence,
                                      l1_lag_samples=0, window_before=2.0, window_after=0.5)
    lag = find_l1_lag_samples(h1z, l1z)
    del h1z, l1z; gc.collect()
    return {'name': name, 'h1': h1, 'l1': l1, 'gps_coalescence': coalescence,
            'l1_lag': lag, 'window_before': 2.0, 'window_after': 0.5}

REAL_EVENTS = [
    resolve_event('GW150914', 1126259446, 1126259462.4232),
    resolve_event('GW151226', 1135136350, 1135136350.648),
    resolve_event('GW170814', 1186741845, 1186741861.530),
    resolve_event('GW170608', 1180922478, 1180922494.49),
    resolve_event('GW170729', 1185389791, 1185389807.3),
    resolve_event('GW170823', 1187529240, 1187529256.5),
    resolve_event('GW170104', 1167559920, 1167559936.6),
]

TRAIN_EVENTS = REAL_EVENTS[:-1]
TEST_EVENT   = REAL_EVENTS[-1]

def create_forecast_sequences(h1_zone, l1_zone, context_len, forecast_len, stride):
    dual = np.stack([h1_zone, l1_zone], axis=-1).astype(np.float32)
    total = context_len + forecast_len
    X, Y = [], []
    for i in range(0, len(dual) - total, stride):
        w = dual[i:i + total]
        X.append(w[:context_len]); Y.append(w[context_len:])
    X = np.nan_to_num(np.array(X, np.float32))
    Y = np.nan_to_num(np.array(Y, np.float32))
    print(f'   Sequences: {X.shape[0]} | X {X.shape} | Y {Y.shape}')
    return X, Y

print(f'\n{len(REAL_EVENTS)} real events resolved (evaluation and controls only, never trained on)')

## SECTION D — Loss Functions & Metrics

In [ ]:
# ============================================================
# CELL 6: FIXED LOSS & METRICS — now covers BOTH H1 and L1
#
# Previous versions of overlap_score_tf/np only computed overlap on
# channel 0 (H1). Since physical_overlap_loss weights this term 10x
# against a 1x MSE term, L1 received essentially no shape-fidelity
# training signal at all — only weak, generic MSE pressure. This is
# the direct cause of L1 predictions collapsing to near-flat lines
# in both models' OOD evaluation. Fixed by averaging overlap across
# both channels.
# ============================================================

def overlap_score_tf(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    overlaps = []
    for ch in range(y_true.shape[-1]):  # loop over H1, L1
        yt = tf.reshape(y_true[:, :, ch], (tf.shape(y_true)[0], -1))
        yp = tf.reshape(y_pred[:, :, ch], (tf.shape(y_pred)[0], -1))
        num  = tf.reduce_sum(yt * yp, axis=1)
        denom = tf.sqrt(tf.reduce_sum(tf.square(yt), axis=1) + 1e-8) * \
                tf.sqrt(tf.reduce_sum(tf.square(yp), axis=1) + 1e-8)
        overlaps.append(tf.reduce_mean(num / (denom + 1e-8)))
    return tf.reduce_mean(tf.stack(overlaps))

def physical_overlap_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, -10.0, 10.0)

    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    overlap = overlap_score_tf(y_true, y_pred)

    true_rms = tf.sqrt(tf.reduce_mean(tf.square(y_true)) + 1e-8)
    pred_rms = tf.sqrt(tf.reduce_mean(tf.square(y_pred)) + 1e-8)
    amp_pen  = tf.abs(pred_rms - true_rms) / (true_rms + 1e-8)

    loss = mse + 10.0 * (1.0 - overlap) + 5.0 * amp_pen
    loss = tf.where(tf.math.is_nan(loss), tf.constant(999.0), loss)
    return loss

def overlap_metric(y_true, y_pred):
    return overlap_score_tf(tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32))

def overlap_score_np(y_true_np, y_pred_np):
    overlaps = []
    for ch in range(y_true_np.shape[-1]):
        yt = y_true_np[..., ch].flatten()
        yp = y_pred_np[..., ch].flatten()
        overlaps.append(np.dot(yt, yp) / (np.linalg.norm(yt) * np.linalg.norm(yp) + 1e-8))
    return float(np.mean(overlaps))

print('Loss functions defined (dual-channel overlap — H1 and L1 both weighted).')

In [ ]:
# ============================================================
# CELL 6b: PER-CHANNEL OVERLAP DIAGNOSTIC (H1 vs L1)
#
# overlap_score_np/overlap_score_tf report the H1+L1 average — useful
# as a single headline number, but it can hide one channel dragging
# the other down (as it did before the dual-channel loss fix). Use
# this whenever you need to see H1 and L1 separately.
# ============================================================

def overlap_score_np_channel(y_true_np, y_pred_np, channel):
    yt = y_true_np[..., channel].flatten()
    yp = y_pred_np[..., channel].flatten()
    return np.dot(yt, yp) / (np.linalg.norm(yt) * np.linalg.norm(yp) + 1e-8)

print('Per-channel diagnostic defined.')

In [ ]:
# ============================================================
# CELL 7: LOSS WEIGHT SENSITIVITY ANALYSIS
#
# Reviewer requirement: show architectural ranking holds
# across different alpha (overlap weight) values.
# Run this AFTER training in Cell 10-12, or use stored results.
# ============================================================

def make_loss_with_alpha(alpha):
    """Factory: returns a loss function with a specific overlap weight."""
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        mse     = tf.reduce_mean(tf.square(y_true - y_pred))
        overlap = overlap_score_tf(y_true, y_pred)
        true_rms = tf.sqrt(tf.reduce_mean(tf.square(y_true), axis=[1,2]) + 1e-8)
        pred_rms = tf.sqrt(tf.reduce_mean(tf.square(y_pred), axis=[1,2]) + 1e-8)
        amp_pen  = tf.reduce_mean(tf.abs(pred_rms - true_rms) / (true_rms + 1e-8))
        return mse + alpha * (1.0 - overlap) + 5.0 * amp_pen
    loss_fn.__name__ = f'loss_alpha_{alpha}'
    return loss_fn

ALPHA_VALUES = [1.0, 5.0, 10.0, 20.0]
print(f'Alpha sensitivity will test: {ALPHA_VALUES}')
print('Run plot_alpha_sensitivity() after all models are trained.')

## SECTION E — Model Architectures

In [ ]:
# ============================================================
# CELL 8: LSTM ENCODER-DECODER (genuine forecasting)
#
# Encoder reads the CONTEXT_LEN observed samples and compresses them
# into a final hidden/cell state. The decoder is seeded ONLY from that
# state — it never sees the target sequence (non-autoregressive,
# single forward pass) — and unrolls for FORECAST_LEN steps to produce
# the prediction. This is the only way the encoder's summary can be
# tested honestly: there's no ground truth for the decoder to peek at.
# ============================================================

def build_lstm_seq2seq(context_len, forecast_len, n_features, hidden_size=64):
    inp = keras.Input(shape=(context_len, n_features), name='input')

    # --- Encoder ---
    x = layers.Conv1D(hidden_size, kernel_size=PATCH*2, strides=PATCH,
                      padding='same', name='patch_embed')(inp)
    x = layers.LSTM(hidden_size, return_sequences=True, name='enc_lstm_1')(x)
    x = layers.Dropout(0.2)(x)
    _, state_h, state_c = layers.LSTM(hidden_size, return_state=True, name='enc_lstm_2')(x)

    # --- Decoder ---
    # RepeatVector broadcasts the encoder's summary vector as a constant
    # input at every forecast timestep; the decoder LSTM's own recurrence
    # (seeded with the encoder's final state) is what lets it produce a
    # non-constant, evolving forecast from that single summary.
    dec_input = layers.RepeatVector(forecast_len // PATCH)(state_h)
    dec = layers.LSTM(hidden_size, return_sequences=True, name='dec_lstm_1')(
        dec_input, initial_state=[state_h, state_c]
    )
    dec = layers.Dropout(0.2)(dec)
    dec = layers.LSTM(hidden_size, return_sequences=True, name='dec_lstm_2')(dec)
    dec = layers.Dropout(0.2)(dec)
    dec = layers.Conv1D(n_features * PATCH, kernel_size=1, name='output_proj')(dec)
    out = layers.Reshape((forecast_len, n_features), name='output')(dec)

    model = keras.Model(inp, out, name='LSTM_Seq2Seq')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=physical_overlap_loss,
        metrics=[overlap_metric]
    )
    return model

print('build_lstm_seq2seq defined.')

In [ ]:
# ============================================================
# CELL 9: TRANSFORMER ENCODER-DECODER (genuine forecasting)
#
# Encoder: same self-attention stack as before, but now only ever sees
# CONTEXT_LEN observed samples.
# Decoder: FORECAST_LEN learned query tokens (one per future timestep),
# independent of the input's content. Each query self-attends to the
# other queries, then cross-attends to the encoded context — it can
# only pull information FROM the context, never from the target. This
# keeps decoding non-autoregressive (one forward pass, no teacher
# forcing) while still forcing genuine forecasting.
# ============================================================

def positional_encoding(seq_len, embed_dim):
    positions = np.arange(seq_len)[:, np.newaxis]
    dims      = np.arange(embed_dim)[np.newaxis, :]
    angles    = positions / np.power(10000, (2*(dims//2)) / embed_dim)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, :, :], tf.float32)

def transformer_block(x, embed_dim, num_heads, ff_dim, dropout_rate=0.2, name_prefix=''):
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim // num_heads,
        dropout=dropout_rate, name=f'{name_prefix}self_attn'
    )(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_out)
    ff = layers.Conv1D(ff_dim, kernel_size=1, activation='relu',
                        kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    ff = layers.Dropout(dropout_rate)(ff)
    ff = layers.Conv1D(embed_dim, kernel_size=1,
                        kernel_regularizer=keras.regularizers.l2(1e-4))(ff)
    ff = layers.Dropout(dropout_rate)(ff)
    x  = layers.LayerNormalization(epsilon=1e-6)(x + ff)
    return x

def build_transformer_seq2seq(context_len, forecast_len, n_features,
                               embed_dim=64, num_heads=4, ff_dim=128,
                               num_enc_layers=2, num_dec_layers=2, dropout=0.2):
    # --- Encoder ---
    inp = keras.Input(shape=(context_len, n_features), name='input')
    x = layers.Conv1D(embed_dim, kernel_size=PATCH*2, strides=PATCH,
                      padding='same', name='patch_embed')(inp)
    x = x + positional_encoding(context_len // PATCH, embed_dim)
    for i in range(num_enc_layers):
        x = transformer_block(x, embed_dim, num_heads, ff_dim, dropout, name_prefix=f'enc{i}_')
    enc_out = x  # (batch, context_len, embed_dim)

    # --- Decoder: learned forecast queries ---
    query_embed_layer = layers.Embedding(forecast_len // PATCH, embed_dim, name='forecast_query_embed')
    query_ids = tf.range(forecast_len // PATCH)
    queries = query_embed_layer(query_ids)                # (forecast_len, embed_dim)
    queries = queries[tf.newaxis, :, :]                    # (1, forecast_len, embed_dim)
    batch_size = tf.shape(inp)[0]
    dec = tf.tile(queries, [batch_size, 1, 1])              # (batch, forecast_len, embed_dim)

    for i in range(num_dec_layers):
        self_attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads,
            dropout=dropout, name=f'dec{i}_self_attn'
        )(dec, dec)
        dec = layers.LayerNormalization(epsilon=1e-6)(dec + self_attn)

        cross_attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads,
            dropout=dropout, name=f'dec{i}_cross_attn'
        )(dec, enc_out)
        dec = layers.LayerNormalization(epsilon=1e-6)(dec + cross_attn)

        ff = layers.Conv1D(ff_dim, kernel_size=1, activation='relu')(dec)
        ff = layers.Dropout(dropout)(ff)
        ff = layers.Conv1D(embed_dim, kernel_size=1)(ff)
        ff = layers.Dropout(dropout)(ff)
        dec = layers.LayerNormalization(epsilon=1e-6)(dec + ff)

    dec = layers.Conv1D(64, kernel_size=1, activation='gelu', name='pre_output',
                         kernel_regularizer=keras.regularizers.l2(1e-4))(dec)
    dec = layers.Dropout(dropout)(dec)
    dec = layers.Conv1D(n_features * PATCH, kernel_size=1, name='output_proj')(dec)
    out = layers.Reshape((forecast_len, n_features), name='output')(dec)

    model = keras.Model(inp, out, name='Transformer_Seq2Seq')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=physical_overlap_loss,
        metrics=[overlap_metric]
    )
    return model

print('build_transformer_seq2seq defined.')

### E1 — Matched parameter budget

Editor pt 4: the architectures must be compared at comparable capacity.
Search only, no training.

In [ ]:
TARGET_PARAMS = 400_000

best_lstm_hs, best_lstm_diff, best_lstm_n = None, float('inf'), None
for hs in [80, 88, 96, 104, 112, 120, 128]:
    m = build_lstm_seq2seq(CONTEXT_LEN, FORECAST_LEN, N_FEATURES, hidden_size=hs)
    n = m.count_params()
    print(f'   LSTM hidden_size={hs:>4} -> {n:,}')
    if abs(n - TARGET_PARAMS) < best_lstm_diff:
        best_lstm_diff, best_lstm_hs, best_lstm_n = abs(n - TARGET_PARAMS), hs, n
    del m
    keras.backend.clear_session()

best_tf_dim, best_tf_diff, best_tf_n = None, float('inf'), None
for ed in [64, 76, 84, 92, 100, 108, 116]:
    m = build_transformer_seq2seq(CONTEXT_LEN, FORECAST_LEN, N_FEATURES,
                                   embed_dim=ed, num_heads=4, ff_dim=ed * 2,
                                   num_enc_layers=2, num_dec_layers=2, dropout=0.2)
    n = m.count_params()
    print(f'   Transformer embed_dim={ed:>4} -> {n:,}')
    if abs(n - TARGET_PARAMS) < best_tf_diff:
        best_tf_diff, best_tf_dim, best_tf_n = abs(n - TARGET_PARAMS), ed, n
    del m
    keras.backend.clear_session()

BEST_LSTM_HS, BEST_TF_DIM = best_lstm_hs, best_tf_dim
mismatch = abs(best_lstm_n - best_tf_n) / max(best_lstm_n, best_tf_n)
print(f'\nChosen: LSTM hidden_size={BEST_LSTM_HS} ({best_lstm_n:,}), '
      f'Transformer embed_dim={BEST_TF_DIM} ({best_tf_n:,})')
print(f'Parameter mismatch: {mismatch*100:.1f}%  ' +
      ('OK' if mismatch < 0.05 else 'WARNING: >5%, widen the search grid'))

## SECTION F — Baselines and training

### F1 — Baseline gate

Run this **before** any long training. If the trained models do not clearly beat
AR-64, the experimental setup is wrong and no amount of training will fix it.

In [ ]:
def overlap_np_batch(Yt, Yp):
    a = Yt.reshape(len(Yt), -1)
    b = Yp.reshape(len(Yp), -1)
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)

def persistence_forecast(X):
    return np.repeat(X[:, -1:, :], FORECAST_LEN, axis=1)

def ar_forecast(X, order=64, limit=200):
    X = X[:limit]
    n, _, c = X.shape
    out = np.zeros((n, FORECAST_LEN, c), np.float32)
    for i in range(n):
        for ch in range(c):
            x = X[i, :, ch].astype(np.float64)
            A = np.stack([x[k:len(x) - order + k] for k in range(order)], 1)
            coef, *_ = np.linalg.lstsq(A, x[order:], rcond=None)
            buf = list(x[-order:])
            for _ in range(FORECAST_LEN):
                buf.append(float(np.dot(coef, buf[-order:])))
            out[i, :, ch] = buf[-FORECAST_LEN:]
    return out

BASELINES = {}
_ar = ar_forecast(X_val)
BASELINES['ar64'] = float(overlap_np_batch(Y_val[:len(_ar)], _ar).mean())
BASELINES['persistence'] = float(overlap_np_batch(Y_val, persistence_forecast(X_val)).mean())
BASELINES['mean_waveform'] = float(overlap_np_batch(
    Y_val, np.repeat(Y_train.mean(0)[None], len(Y_val), 0)).mean())

print('BASELINE GATE (validation set)')
for k, v in BASELINES.items():
    print(f'   {k:15s} {v:.4f}')
print(f'\nTrained models must clearly exceed ar64 = {BASELINES["ar64"]:.4f}')

In [ ]:
BATCH_SIZE = 64
EPOCHS = 100

class WarmupLR(keras.callbacks.Callback):
    def __init__(self, target_lr=1e-4, warmup_epochs=10):
        super().__init__()
        self.target_lr = target_lr
        self.warmup_epochs = warmup_epochs
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            keras.backend.set_value(
                self.model.optimizer.learning_rate,
                self.target_lr * (epoch + 1) / self.warmup_epochs)

def get_callbacks(model_name, run_tag):
    d = os.path.join('ckpt', run_tag)
    os.makedirs(d, exist_ok=True)
    return [
        WarmupLR(target_lr=1e-4, warmup_epochs=10),
        EarlyStopping(monitor='val_loss', patience=15,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8,
                          min_lr=1e-7, verbose=1),
        ModelCheckpoint(filepath=os.path.join(d, f'{model_name}_best.keras'),
                        monitor='val_loss', save_best_only=True, verbose=1),
    ]

print(f'Training config: batch={BATCH_SIZE}, max_epochs={EPOCHS}')
print('Callbacks: Warmup(10ep), EarlyStopping(patience=15), ReduceLROnPlateau(patience=8)')

### F2 — Multi-seed training

Editor pt 4 and 8. Every architecture number in the manuscript comes from this
cell: mean +/- std across 5 seeds, with a paired t-test. A single seed cannot
support a ranking claim, which is what sank v1.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]

def run_seed(builder, kwargs, seed, tag, epochs=EPOCHS):
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model = builder(CONTEXT_LEN, FORECAST_LEN, N_FEATURES, **kwargs)
    t0 = time.time()
    hist = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                     epochs=epochs, batch_size=BATCH_SIZE,
                     callbacks=get_callbacks(tag, f'{tag}_s{seed}'), verbose=2)
    train_time = time.time() - t0
    rec = {'seed': seed, 'tag': tag, 'history': hist.history,
           'params': model.count_params(), 'train_time_s': train_time,
           'epochs_run': len(hist.history['loss']),
           'val_overlap': float(overlap_np_batch(
               Y_val, model.predict(X_val, batch_size=128, verbose=0)).mean())}
    for k in OOD_SETS:
        Xo, Yo, _ = ood_data[k]
        rec[k] = float(overlap_np_batch(Yo, model.predict(Xo, batch_size=128, verbose=0)).mean())
    line = (f"{tag} seed {seed}: val {rec['val_overlap']:.4f} | " +
            ' | '.join(f'{k} {rec[k]:.4f}' for k in OOD_SETS) +
            f" | {rec['epochs_run']} ep | {train_time/60:.1f} min")
    print('  ' + line)
    with open('training_log.txt', 'a') as f:
        f.write('FINAL ' + line + '\n')
    del model
    return rec

LSTM_KW = dict(hidden_size=BEST_LSTM_HS)
TF_KW   = dict(embed_dim=BEST_TF_DIM, num_heads=4, ff_dim=BEST_TF_DIM * 2,
               num_enc_layers=2, num_dec_layers=2, dropout=0.2)

results = {'lstm': [], 'transformer': []}
for s in SEEDS:
    results['lstm'].append(run_seed(build_lstm_seq2seq, LSTM_KW, s, 'lstm'))
    results['transformer'].append(run_seed(build_transformer_seq2seq, TF_KW, s, 'transformer'))

with open('multiseed_results_final.pkl', 'wb') as f:
    pickle.dump(results, f)

from scipy.stats import ttest_rel
print('\n' + '=' * 70)
print('FINAL MULTI-SEED (warmup=10, patience=15, 100 epochs, matched params)')
print('=' * 70)
for metric in ['val_overlap'] + OOD_SETS:
    a = [r[metric] for r in results['lstm']]
    b = [r[metric] for r in results['transformer']]
    t, p = ttest_rel(a, b)
    print(f'{metric:12s} LSTM {np.mean(a):.4f}+/-{np.std(a):.4f} | '
          f'TF {np.mean(b):.4f}+/-{np.std(b):.4f} | p={p:.4f} '
          f'({"significant" if p < 0.05 else "NOT significant"})')
print('\nLSTM seeds: ', [round(r['val_overlap'], 4) for r in results['lstm']])
print('TF seeds:   ', [round(r['val_overlap'], 4) for r in results['transformer']])
print(f'Baseline ar64 = {BASELINES["ar64"]:.4f}')

In [ ]:
class WarmupLR(keras.callbacks.Callback):
    def __init__(self, target_lr=1e-4, warmup_epochs=5):
        super().__init__()
        self.target_lr = target_lr
        self.warmup_epochs = warmup_epochs
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            keras.backend.set_value(
                self.model.optimizer.learning_rate,
                self.target_lr * (epoch + 1) / self.warmup_epochs)

SEED_TO_FIX = 1

for label, warm, clip in [('warmup10', 10, None),
                          ('warmup10+clip1', 10, 1.0),
                          ('warmup5+clip1', 5, 1.0)]:
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED_TO_FIX)
    m = build_transformer_seq2seq(CONTEXT_LEN, FORECAST_LEN, N_FEATURES, **TF_KW)
    opt = (keras.optimizers.Adam(1e-4, clipnorm=clip) if clip
           else keras.optimizers.Adam(1e-4))
    m.compile(optimizer=opt, loss=physical_overlap_loss, metrics=[overlap_metric])
    cbs = [WarmupLR(target_lr=1e-4, warmup_epochs=warm),
           EarlyStopping(monitor='val_loss', patience=15,
                         restore_best_weights=True, verbose=0),
           ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                             min_lr=1e-7, verbose=0)]
    m.fit(X_train, Y_train, validation_data=(X_val, Y_val),
          epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cbs, verbose=0)
    ov = float(overlap_np_batch(Y_val, m.predict(X_val, batch_size=128, verbose=0)).mean())
    line = f'seed {SEED_TO_FIX} {label:16s} val overlap {ov:.4f}'
    print(line)
    with open('training_log.txt', 'a') as f:
        f.write(line + '\n')
    del m

## SECTION G — Evaluation

In [ ]:
from scipy.signal import hilbert

def phase_error(Yt, Yp, ch=0):
    errs = []
    for i in range(len(Yt)):
        pt = np.angle(hilbert(Yt[i, :, ch]))
        pp = np.angle(hilbert(Yp[i, :, ch]))
        errs.append(np.mean(np.abs(np.angle(np.exp(1j * (pt - pp))))))
    return np.array(errs)

def amplitude_ratio(Yt, Yp):
    rt = np.sqrt(np.mean(Yt ** 2, axis=(1, 2))) + 1e-12
    rp = np.sqrt(np.mean(Yp ** 2, axis=(1, 2)))
    return rp / rt

def rank_by_energy(Y, k=5):
    e = np.sqrt(np.mean(Y ** 2, axis=(1, 2)))
    return np.argsort(e)[::-1][:k]

def evaluate_model(model, X, Y, name, num_samples=5, batch_size=128, plot=True):
    P = model.predict(X, batch_size=batch_size, verbose=0)
    ov = overlap_np_batch(Y, P)
    ph = phase_error(Y, P)
    ar = amplitude_ratio(Y, P)
    mse = np.mean((Y - P) ** 2, axis=(1, 2))
    res = {'name': name, 'preds': P, 'overlaps': ov, 'mses': mse,
           'mean_overlap': float(ov.mean()), 'std_overlap': float(ov.std()),
           'mean_mse': float(mse.mean()),
           'phase_err': ph, 'mean_phase_err': float(ph.mean()),
           'amp_ratio': ar, 'mean_amp_ratio': float(ar.mean()),
           'overlap_h1': float(overlap_np_batch(Y[..., 0:1], P[..., 0:1]).mean()),
           'overlap_l1': float(overlap_np_batch(Y[..., 1:2], P[..., 1:2]).mean())}
    print(f"{name}: overlap {res['mean_overlap']:.4f}+/-{res['std_overlap']:.4f} | "
          f"phase {res['mean_phase_err']:.4f} rad | amp {res['mean_amp_ratio']:.3f} | "
          f"H1 {res['overlap_h1']:.4f} L1 {res['overlap_l1']:.4f}")
    if plot:
        idx = rank_by_energy(Y, num_samples)
        fig, axes = plt.subplots(num_samples, 2, figsize=(15, 2.6 * num_samples))
        t = np.arange(FORECAST_LEN) / FS
        for r, i in enumerate(idx):
            for ch, det in enumerate(['H1 (Hanford)', 'L1 (Livingston)']):
                ax = axes[r, ch]
                ax.plot(t, Y[i, :, ch], 'k-', lw=1.0, label='True (noiseless)')
                ax.plot(t, P[i, :, ch], 'r--', lw=1.0, label='Prediction')
                ax.set_title(f'Sample {i} - {det} | Overlap {ov[i]:.4f}', fontsize=9)
                ax.set_xlabel('Time (s)'); ax.set_ylabel('Strain')
                if r == 0 and ch == 0:
                    ax.legend(fontsize=7)
                ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{name}_predictions.png', dpi=200, bbox_inches='tight')
        plt.show()
    return res

print('Evaluation utilities defined (samples ranked by target energy, not chronological).')

In [ ]:
lstm_model = keras.models.load_model(
    os.path.join('ckpt', 'lstm_s0', 'lstm_best.keras'), compile=False)
transformer_model = keras.models.load_model(
    os.path.join('ckpt', 'transformer_s0', 'transformer_best.keras'), compile=False)

print('LSTM       ', f'{lstm_model.count_params():,}')
print('Transformer', f'{transformer_model.count_params():,}')
assert abs(lstm_model.count_params() - transformer_model.count_params()) / \
       max(lstm_model.count_params(), transformer_model.count_params()) < 0.05, \
       'parameter budgets differ by more than 5%'

_lv = float(overlap_np_batch(Y_val, lstm_model.predict(X_val, batch_size=128, verbose=0)).mean())
_tv = float(overlap_np_batch(Y_val, transformer_model.predict(X_val, batch_size=128, verbose=0)).mean())
print(f'reloaded val overlap  LSTM {_lv:.4f} (seed-0 recorded 0.8244)')
print(f'reloaded val overlap  TF   {_tv:.4f} (seed-0 recorded 0.7114)')
assert abs(_lv - 0.8244) < 0.02 and abs(_tv - 0.7114) < 0.02, 'reloaded models do not match training'

print('\n--- In-distribution validation ---')
results_lstm_id  = evaluate_model(lstm_model, X_val, Y_val, 'LSTM_val')
results_trans_id = evaluate_model(transformer_model, X_val, Y_val, 'Transformer_val')

print('\n--- Out-of-distribution ---')
ood_results = {}
for k in OOD_SETS:
    Xo, Yo, _ = ood_data[k]
    ood_results[k] = {
        'lstm': evaluate_model(lstm_model, Xo, Yo, f'LSTM_{k}', plot=(k == 'ood_q')),
        'transformer': evaluate_model(transformer_model, Xo, Yo, f'Transformer_{k}',
                                       plot=(k == 'ood_q')),
    }

results_lstm = ood_results['ood_q']['lstm']
results_transformer = ood_results['ood_q']['transformer']

### G1 — Overlap versus forecast horizon

New contribution, and the direct answer to Editor pt 6 and 8: it shows *where*
the task stops being solvable instead of asserting an architectural advantage.

In [ ]:
HORIZONS = [8, 16, 24, 32]
def overlap_to_horizon(model, X, Y, horizons):
    P = model.predict(X, batch_size=128, verbose=0)
    return [float(overlap_np_batch(Y[:, :h], P[:, :h]).mean()) for h in horizons]

h_lstm  = overlap_to_horizon(lstm_model, X_val, Y_val, HORIZONS)
h_trans = overlap_to_horizon(transformer_model, X_val, Y_val, HORIZONS)

fig, ax = plt.subplots(figsize=(8, 5))
ms = [h / FS * 1000 for h in HORIZONS]
ax.plot(ms, h_lstm, 'o-', label='LSTM')
ax.plot(ms, h_trans, 's-', label='Transformer')
ax.axhline(BASELINES['ar64'], color='gray', ls='--', label='AR-64 baseline')
ax.set_xlabel('Forecast horizon (ms)'); ax.set_ylabel('Overlap')
ax.set_title('Forecast fidelity vs prediction horizon', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('overlap_vs_horizon.png', dpi=300, bbox_inches='tight')
plt.show()

for h, a, b in zip(ms, h_lstm, h_trans):
    print(f'  {h:6.0f} ms   LSTM {a:.4f}   Transformer {b:.4f}')

In [ ]:
Xm_tr, Ym_tr, _ = inject(bank['bank_train'], noise_pool, 21, mode='merger')
Xm_va, Ym_va, _ = inject(bank['bank_val'],   noise_pool, 22, mode='merger')
keras.backend.clear_session()
tf.keras.utils.set_random_seed(0)
mm = build_lstm_seq2seq(CONTEXT_LEN, FORECAST_LEN, N_FEATURES, **LSTM_KW)
mm.fit(Xm_tr, Ym_tr, validation_data=(Xm_va, Ym_va), epochs=25,
       batch_size=BATCH_SIZE, callbacks=get_callbacks('lstm', 'merger_mode'), verbose=2)
ov_m = overlap_np_batch(Ym_va, mm.predict(Xm_va, batch_size=128, verbose=0))
print(f'MERGER-INCLUSIVE overlap {ov_m.mean():.4f} +/- {ov_m.std():.4f}')
print(f'INSPIRAL-ONLY    overlap {results_lstm_id["mean_overlap"]:.4f}')
print(f'AR-64 baseline   {BASELINES["ar64"]:.4f}')

## SECTION H — Ablation Studies

In [ ]:
# ============================================================
# CELL 15: ABLATION 1 — Single vs Dual Detector
# Updated for the seq2seq forecast architecture (CONTEXT_LEN -> FORECAST_LEN).
# Previously called an old build_transformer(SEQ_OUT, ...) API left over
# from before the Phase-2 redesign; SEQ_OUT no longer exists. Now uses
# build_lstm_seq2seq / build_transformer_seq2seq with n_features=1.
# ============================================================

print('ABLATION 1: Single (H1 only) vs Dual (H1+L1) detector input')
print('='*60)

X_train_h1 = X_train[..., 0:1]
Y_train_h1 = Y_train[..., 0:1]
X_val_h1   = X_val[..., 0:1]
Y_val_h1   = Y_val[..., 0:1]
X_test_h1  = X_test[..., 0:1]
Y_test_h1  = Y_test[..., 0:1]

lstm_h1only = build_lstm_seq2seq(CONTEXT_LEN, FORECAST_LEN, n_features=1, hidden_size=96)
trans_h1only = build_transformer_seq2seq(
    CONTEXT_LEN, FORECAST_LEN, n_features=1,
    embed_dim=96, num_heads=4, ff_dim=192,
    num_enc_layers=2, num_dec_layers=2, dropout=0.2
)

print('Training H1-only LSTM...')
hist_lstm_h1 = lstm_h1only.fit(
    X_train_h1, Y_train_h1,
    validation_data=(X_val_h1, Y_val_h1),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks('lstm_h1only', 'ablation_h1only'), verbose=0
)

print('Training H1-only Transformer...')
hist_trans_h1 = trans_h1only.fit(
    X_train_h1, Y_train_h1,
    validation_data=(X_val_h1, Y_val_h1),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks('transformer_h1only', 'ablation_h1only'), verbose=0
)

def get_overlap_on_test(model, X, Y, n=None):
    n = len(X) if n is None else min(n, len(X))
    preds = model.predict(X[:n], verbose=0)
    return np.mean([overlap_score_np(Y[i], preds[i]) for i in range(n)])

# Compare against the CURRENT full dual-channel models (already trained
# on the 6-event set, in memory as lstm_model / transformer_model)
ov_lstm_h1    = get_overlap_on_test(lstm_h1only,      X_test_h1, Y_test_h1)
ov_trans_h1   = get_overlap_on_test(trans_h1only,     X_test_h1, Y_test_h1)
ov_lstm_dual  = get_overlap_on_test(lstm_model,        X_test,    Y_test)
ov_trans_dual = get_overlap_on_test(transformer_model, X_test,    Y_test)

print(f'\nABLATION RESULTS (full OOD test set, {len(X_test)} sequences):')
print(f'  LSTM        H1-only: {ov_lstm_h1:.4f}')
print(f'  LSTM        H1+L1:   {ov_lstm_dual:.4f}  ({(ov_lstm_dual-ov_lstm_h1)*100:+.2f} pts)')
print(f'  Transformer H1-only: {ov_trans_h1:.4f}')
print(f'  Transformer H1+L1:   {ov_trans_dual:.4f}  ({(ov_trans_dual-ov_trans_h1)*100:+.2f} pts)')

fig, ax = plt.subplots(figsize=(9, 5))
models_ab = ['LSTM\nH1 only', 'LSTM\nH1+L1', 'Transformer\nH1 only', 'Transformer\nH1+L1']
values_ab = [ov_lstm_h1, ov_lstm_dual, ov_trans_h1, ov_trans_dual]
colors_ab = ['#90CAF9', '#1565C0', '#EF9A9A', '#B71C1C']
bars = ax.bar(models_ab, values_ab, color=colors_ab, edgecolor='black', linewidth=1.2)
ax.set_ylabel('Overlap Score (OOD test)', fontsize=12, fontweight='bold')
ax.set_title('Ablation 1: Single vs Dual Detector Input', fontsize=14, fontweight='bold')
ax.bar_label(bars, fmt='%.4f')
ax.grid(alpha=0.3, ls='--', axis='y')
plt.tight_layout()
plt.savefig('ablation1_single_vs_dual.png', dpi=300, bbox_inches='tight')
print('Saved: ablation1_single_vs_dual.png')
plt.show()

In [ ]:
# ============================================================
# CELL 16: ABLATION 2 — Loss Weight (Alpha) Sensitivity
# Updated for the seq2seq forecast architecture. Previously called
# build_transformer(SEQ_OUT, ...) / build_lstm_stable, both leftover
# from before the Phase-2 redesign; SEQ_OUT no longer exists.
# ============================================================

print('ABLATION 2: Loss weight alpha sensitivity')
print('='*60)

alpha_results = {'lstm': [], 'transformer': []}

# The main models train with a FIXED overlap weight of 10.0, not alpha=0.5
# (this is the Eq. 13 discrepancy flagged for manuscript correction).
# Make sure that operating point is included and marked, so the plot
# honestly shows where the headline results sit.
MAIN_MODEL_WEIGHT = 10.0
if MAIN_MODEL_WEIGHT not in ALPHA_VALUES:
    ALPHA_VALUES = sorted(ALPHA_VALUES + [MAIN_MODEL_WEIGHT])
    print(f"Added main model's actual training weight ({MAIN_MODEL_WEIGHT}) to sweep: {ALPHA_VALUES}")

for alpha in ALPHA_VALUES:
    print(f'\n--- Alpha = {alpha} ---')

    tf.keras.utils.set_random_seed(0)

    loss_fn = make_loss_with_alpha(alpha)

    m_lstm = build_lstm_seq2seq(CONTEXT_LEN, FORECAST_LEN, N_FEATURES, **LSTM_KW)
    m_lstm.compile(optimizer=keras.optimizers.Adam(1e-4),
                   loss=loss_fn, metrics=[overlap_metric])
    m_lstm.fit(X_train, Y_train, validation_data=(X_val, Y_val),
               epochs=20, batch_size=BATCH_SIZE,
               callbacks=get_callbacks(f'lstm_alpha_{alpha}', f'alpha_{alpha}'),
               verbose=0)

    tf.keras.utils.set_random_seed(0)

    m_trans = build_transformer_seq2seq(CONTEXT_LEN, FORECAST_LEN, N_FEATURES, **TF_KW)
    m_trans.compile(optimizer=keras.optimizers.Adam(1e-4),
                    loss=loss_fn, metrics=[overlap_metric])
    m_trans.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                epochs=20, batch_size=BATCH_SIZE,
                callbacks=get_callbacks(f'transformer_alpha_{alpha}', f'alpha_{alpha}'),
                verbose=0)

    ov_l = get_overlap_on_test(m_lstm,  X_test, Y_test)
    ov_t = get_overlap_on_test(m_trans, X_test, Y_test)
    alpha_results['lstm'].append(ov_l)
    alpha_results['transformer'].append(ov_t)
    print(f'   LSTM: {ov_l:.4f} | Transformer: {ov_t:.4f}')

    del m_lstm, m_trans
    gc.collect()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ALPHA_VALUES, alpha_results['lstm'],        'bo-', lw=2.5, ms=8, label='LSTM Baseline')
ax.plot(ALPHA_VALUES, alpha_results['transformer'], 'r^-', lw=2.5, ms=8, label='Improved Transformer')
ax.axvline(MAIN_MODEL_WEIGHT, color='purple', ls=':', lw=2,
           label=f'Main model operating point (w={MAIN_MODEL_WEIGHT})')
ax.set_xlabel('Overlap Loss Weight (α)', fontsize=12, fontweight='bold')
ax.set_ylabel('Overlap Score (OOD test, full set)', fontsize=12, fontweight='bold')
ax.set_title('Alpha Sensitivity: Does the LSTM/Transformer Ranking Hold?', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ablation_alpha_sensitivity.png', dpi=300, bbox_inches='tight')
print('\nSaved: ablation_alpha_sensitivity.png')
plt.show()

print(f'\nRESULTS TABLE:')
for a, l, t in zip(ALPHA_VALUES, alpha_results['lstm'], alpha_results['transformer']):
    tag = '  <-- main model weight' if a == MAIN_MODEL_WEIGHT else ''
    print(f'  alpha={a:>5}: LSTM={l:.4f} | Transformer={t:.4f}{tag}')

In [ ]:
# ============================================================
# CELL 17: ABLATION 3 — Phase Error Analysis
#
# Uses Hilbert transform to compute instantaneous phase.
# Addresses Reviewer 2 point 3 (overlap vs SNR contradiction).
# ============================================================

print('ABLATION 3: Phase error analysis (Hilbert transform)')

n_phase = min(20, len(X_test))
pred_trans = transformer_model.predict(X_test[:n_phase], verbose=0)
pred_lstm  = lstm_model.predict(X_test[:n_phase], verbose=0)

def compute_phase_error(true_np, pred_np):
    """Mean absolute phase error in radians using Hilbert transform."""
    errors = []
    for i in range(len(true_np)):
        yt = true_np[i, :, 0]   # H1 channel
        yp = pred_np[i, :, 0]
        phase_true = np.angle(hilbert(yt))
        phase_pred = np.angle(hilbert(yp))
        phase_diff = np.abs(np.angle(np.exp(1j * (phase_true - phase_pred))))
        errors.append(np.mean(phase_diff))
    return np.array(errors)

phase_err_trans = compute_phase_error(Y_test[:n_phase], pred_trans)
phase_err_lstm  = compute_phase_error(Y_test[:n_phase], pred_lstm)

# Plot phase error over time for best sample
best_idx = np.argsort(phase_err_trans)[len(phase_err_trans) // 2]
yt_best = Y_test[best_idx, :, 0]
pt_best = pred_trans[best_idx, :, 0]
pl_best = pred_lstm[best_idx, :, 0]

phase_true  = np.angle(hilbert(yt_best))
phase_trans = np.angle(hilbert(pt_best))
phase_lstm  = np.angle(hilbert(pl_best))
t_ax = np.arange(len(yt_best)) / FS

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(t_ax, np.abs(np.unwrap(phase_true - phase_trans)),
        'r-', lw=2, label=f'Transformer (mean={np.mean(phase_err_trans):.3f} rad)')
ax.plot(t_ax, np.abs(np.unwrap(phase_true - phase_lstm)),
        'b-', lw=2, label=f'LSTM (mean={np.mean(phase_err_lstm):.3f} rad)')
ax.set_xlabel('Time (s)', fontsize=12); ax.set_ylabel('|Phase Error| (rad)', fontsize=12)
ax.set_title('Instantaneous Phase Error (H1)', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.boxplot([phase_err_trans, phase_err_lstm],
           labels=['Transformer', 'LSTM'],
           patch_artist=True,
           boxprops=dict(facecolor='lightcoral'),
           medianprops=dict(color='black', lw=2))
ax.set_ylabel('Mean Phase Error (rad)', fontsize=12)
ax.set_title('Phase Error Distribution Across Test Samples', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Phase Coherence Analysis — Addressing Overlap vs SNR Discrepancy',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('phase_error_analysis.png', dpi=300, bbox_inches='tight')
print('Saved: phase_error_analysis.png')
plt.show()

print(f'\nPHASE ERROR SUMMARY:')
print(f'  Transformer: {np.mean(phase_err_trans):.4f} ± {np.std(phase_err_trans):.4f} rad')
print(f'  LSTM:        {np.mean(phase_err_lstm):.4f} ± {np.std(phase_err_lstm):.4f} rad')

## SECTION I — Final Comparison Table

In [ ]:
@tf.function(reduce_retracing=True)
def _infer(model, x):
    return model(x, training=False)

def benchmark_latency(model, X, n_runs=200):
    x = tf.constant(X)
    for _ in range(20):
        _ = _infer(model, x)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = _infer(model, x)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))

print('Logical GPU devices:', tf.config.list_logical_devices('GPU'))
X_single = X_val[0:1]
lat_lstm_mean,  lat_lstm_std  = benchmark_latency(lstm_model, X_single)
lat_trans_mean, lat_trans_std = benchmark_latency(transformer_model, X_single)

def agg(model_key, metric):
    v = [r[metric] for r in results[model_key]]
    return f'{np.mean(v):.4f} +/- {np.std(v):.4f}'

comparison = pd.DataFrame({
    'Model': ['LSTM', 'Transformer'],
    'Overlap (val, 5 seeds)': [agg('lstm', 'val_overlap'), agg('transformer', 'val_overlap')],
    'Overlap (OOD q)':        [agg('lstm', 'ood_q'), agg('transformer', 'ood_q')],
    'Overlap (OOD spin)':     [agg('lstm', 'ood_spin'), agg('transformer', 'ood_spin')],
    'Overlap (OOD mass)':     [agg('lstm', 'ood_mass'), agg('transformer', 'ood_mass')],
    'MSE (OOD q)':            [f"{results_lstm['mean_mse']:.6f}",
                               f"{results_transformer['mean_mse']:.6f}"],
    'Phase err (rad)':        [f"{results_lstm['mean_phase_err']:.4f}",
                               f"{results_transformer['mean_phase_err']:.4f}"],
    'Amp ratio':              [f"{results_lstm['mean_amp_ratio']:.3f}",
                               f"{results_transformer['mean_amp_ratio']:.3f}"],
    'Latency (ms)':           [f'{lat_lstm_mean:.1f} +/- {lat_lstm_std:.1f}',
                               f'{lat_trans_mean:.1f} +/- {lat_trans_std:.1f}'],
    'Parameters':             [f'{lstm_model.count_params():,}',
                               f'{transformer_model.count_params():,}'],
})

print('\n' + '=' * 100)
print('FINAL COMPARISON')
print('=' * 100)
print(comparison.to_string(index=False))
comparison.to_csv('model_comparison_final.csv', index=False)

print('\nReporting notes for the manuscript:')
print(f'  Hardware:   NVIDIA RTX 3060, TensorFlow {tf.__version__}, batch_size=1')
print('  Latency measured with a traced model call, not Keras .predict()')
print('  No TensorRT / FlashAttention optimization applied - state this explicitly')
print(f'  Bank mode:  {BANK_MODE}')
print(f'  Context {CONTEXT_LEN/FS:.2f}s -> forecast {FORECAST_LEN/FS:.2f}s at {FS} Hz')
print(f'  Baselines:  AR-64 {BASELINES["ar64"]:.4f}, persistence {BASELINES["persistence"]:.4f}')

## SECTION J — Robustness Controls

Neither model should produce a confident waveform from pure noise or from an
instrumental glitch. Real strain is resampled to the model rate before slicing.

In [ ]:
def noise_only_contexts(pool, n, seed):
    r = np.random.RandomState(seed)
    X = pool[r.randint(0, len(pool), n)][:, :CONTEXT_LEN, :].copy()
    m = X.std(axis=(1, 2), keepdims=True) + 1e-8
    return (X / m).astype(np.float32)

def inject_glitch(X, f0, q, amplitude, seed):
    r = np.random.RandomState(seed)
    out = X.copy()
    t = np.arange(CONTEXT_LEN) / FS
    for i in range(len(out)):
        t0 = t[int(CONTEXT_LEN * r.uniform(0.55, 0.95))]
        tau = q / (2 * np.pi * f0)
        env = np.exp(-((t - t0) ** 2) / (2 * tau ** 2))
        for ch in range(N_FEATURES):
            ph = r.uniform(0, 2 * np.pi)
            out[i, :, ch] += amplitude * np.std(X[i, :, ch]) * env * np.sin(2 * np.pi * f0 * (t - t0) + ph)
    m = out.std(axis=(1, 2), keepdims=True) + 1e-8
    return (out / m).astype(np.float32)

def pred_rms(model, X):
    P = model.predict(X, batch_size=128, verbose=0)
    return np.sqrt(np.mean(P ** 2, axis=(1, 2)))

N_CONTROL = 600
X_noise_ctl = noise_only_contexts(noise_pool, N_CONTROL, 3)
GLITCH_CONFIGS = [
    {'name': 'low_f_broad',  'f0': 40.0,  'q': 4.0,  'amplitude': 3.0},
    {'name': 'mid_f_narrow', 'f0': 150.0, 'q': 10.0, 'amplitude': 3.0},
    {'name': 'high_f_sharp', 'f0': 300.0, 'q': 6.0,  'amplitude': 4.0},
]

control_results = {}
for mname, model in [('LSTM', lstm_model), ('Transformer', transformer_model)]:
    ref = np.median(pred_rms(model, X_val[:N_CONTROL]))
    row = {'signal_rms': float(ref),
           'noise_rms': float(np.median(pred_rms(model, X_noise_ctl)))}
    for cfg in GLITCH_CONFIGS:
        Xg = inject_glitch(X_noise_ctl, cfg['f0'], cfg['q'], cfg['amplitude'], 7)
        row[cfg['name']] = float(np.median(pred_rms(model, Xg)))
    control_results[mname] = row

print('=' * 78)
print('ROBUSTNESS CONTROLS - median predicted amplitude, as a fraction of')
print('the amplitude the same model produces on genuine injections.')
print('A model that hallucinates signal from noise or glitches scores near 1.0.')
print('=' * 78)
for mname, r in control_results.items():
    print(f'\n{mname}  (signal reference RMS {r["signal_rms"]:.4f})')
    print(f'   pure noise      {r["noise_rms"]/r["signal_rms"]:.3f}')
    for cfg in GLITCH_CONFIGS:
        print(f'   {cfg["name"]:15s} {r[cfg["name"]]/r["signal_rms"]:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
labels = ['pure noise'] + [c['name'] for c in GLITCH_CONFIGS]
x = np.arange(len(labels))
for i, (mname, r) in enumerate(control_results.items()):
    vals = [r['noise_rms'] / r['signal_rms']] + \
           [r[c['name']] / r['signal_rms'] for c in GLITCH_CONFIGS]
    ax.bar(x + i * 0.35 - 0.175, vals, width=0.32, label=mname)
ax.axhline(1.0, color='red', ls='--', lw=1.2, label='signal-level output (bad)')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel('Predicted amplitude / signal-case amplitude')
ax.set_title('Robustness controls - lower is better', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('robustness_controls.png', dpi=300, bbox_inches='tight')
plt.show()

### J1 — Real-event evaluation

The simulation-to-real gap. Report this honestly as its own row; a drop here is
expected and is itself a finding.

In [ ]:
templates = np.load('event_templates.npz')
TPL_STRIDE, TPL_THRESH, MAX_LAG = 4, 0.15, 64

def align_template(ch, tpl, max_lag=MAX_LAG):
    best_lag, best_sgn, best_abs = 0, 1.0, -1.0
    for lag in range(-max_lag, max_lag + 1):
        d = float(np.dot(ch, np.roll(tpl, lag)))
        if abs(d) > best_abs:
            best_lag, best_sgn, best_abs = lag, (1.0 if d >= 0 else -1.0), abs(d)
    return best_lag, best_sgn

def select_windows(tpl, n_total):
    peak = np.max(np.abs(tpl))
    c1s = np.arange(CONTEXT_LEN, n_total - FORECAST_LEN + 1, TPL_STRIDE)
    return [int(c) for c in c1s
            if np.sqrt(np.mean(tpl[c:c + FORECAST_LEN] ** 2)) > TPL_THRESH * peak]

def build_X(h1m, l1m, keep):
    dual = np.stack([h1m, l1m], -1).astype(np.float32)
    X = np.stack([dual[c - CONTEXT_LEN:c] for c in keep]).astype(np.float32)
    m = X.std(axis=(1, 2), keepdims=True) + 1e-8
    return (X / m).astype(np.float32)

def build_Y(tpl, keep, sgn):
    return (np.stack([tpl[c:c + FORECAST_LEN] for c in keep]) * sgn).astype(np.float32)

def ov1(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

real_results = {}
for ev in REAL_EVENTS:
    name = ev['name']
    if name not in templates.files:
        continue
    print(f"\n--- {name} ---")
    gps_start = get_event_params(ev['h1'], ev['l1'], ev['gps_coalescence'],
                                 ev['l1_lag'], ev['window_before'], ev['window_after'])
    h1z, l1z, _ = load_merger_window(ev['h1'], ev['l1'], gps_start,
                                     ev['gps_coalescence'], ev['l1_lag'],
                                     ev['window_before'], ev['window_after'])
    h1m, l1m = to_model_rate(h1z), to_model_rate(l1z)
    tpl = templates[name]
    n = min(len(h1m), len(l1m), len(tpl))
    h1m, l1m, tpl = h1m[:n], l1m[:n], tpl[:n]

    lag_h1, sgn_h1 = align_template(h1m, tpl)
    lag_l1, sgn_l1 = align_template(l1m, tpl)
    sep_ms = abs(lag_h1 - lag_l1) / FS * 1000
    print(f"   fitted lag  H1 {lag_h1/FS*1000:+.1f} ms (sgn {sgn_h1:+.0f})  "
          f"L1 {lag_l1/FS*1000:+.1f} ms (sgn {sgn_l1:+.0f})  | separation {sep_ms:.1f} ms"
          + ("  <-- EXCEEDS 10 ms, suspect" if sep_ms > 10.0 else ""))

    tpl_h1, tpl_l1 = np.roll(tpl, lag_h1), np.roll(tpl, lag_l1)
    keep = select_windows(tpl_h1, n)
    if not keep:
        print('   no windows with template signal, skipping')
        continue
    X = build_X(h1m, l1m, keep)
    Y_h1 = build_Y(tpl_h1, keep, sgn_h1)
    Y_l1 = build_Y(tpl_l1, keep, sgn_l1)

    row = {'n_windows': len(keep), 'lag_h1_ms': lag_h1 / FS * 1000,
           'lag_l1_ms': lag_l1 / FS * 1000, 'sep_ms': sep_ms}
    for mname, model in [('lstm', lstm_model), ('transformer', transformer_model)]:
        P = model.predict(X, batch_size=128, verbose=0)
        h1_ov = float(np.mean([ov1(P[i, :, 0], Y_h1[i]) for i in range(len(keep))]))
        l1_ov = float(np.mean([ov1(P[i, :, 1], Y_l1[i]) for i in range(len(keep))]))
        row[mname] = {'h1': h1_ov, 'l1': l1_ov}
    real_results[name] = row
    print(f"   windows {len(keep)}")
    print(f"   LSTM        H1 {row['lstm']['h1']:+.4f}  L1 {row['lstm']['l1']:+.4f}")
    print(f"   Transformer H1 {row['transformer']['h1']:+.4f}  L1 {row['transformer']['l1']:+.4f}")
    del h1z, l1z, X, Y_h1, Y_l1; gc.collect()

print('\n' + '=' * 82)
print('REAL EVENTS vs ALIGNED IMRPhenomD TEMPLATE  (H1 is the primary number)')
print('=' * 82)
print(f"{'Event':10s} {'N':>5s} {'sep ms':>7s} {'LSTM H1':>9s} {'LSTM L1':>9s} {'TF H1':>9s} {'TF L1':>9s}")
for nm, r in real_results.items():
    print(f"{nm:10s} {r['n_windows']:5d} {r['sep_ms']:7.1f} {r['lstm']['h1']:+9.4f} "
          f"{r['lstm']['l1']:+9.4f} {r['transformer']['h1']:+9.4f} {r['transformer']['l1']:+9.4f}")
ok = [r for r in real_results.values() if r['sep_ms'] <= 10.0]
if ok:
    print(f"\nMean H1 (events with physical H1-L1 separation, n={len(ok)}): "
          f"LSTM {np.mean([r['lstm']['h1'] for r in ok]):.4f} | "
          f"Transformer {np.mean([r['transformer']['h1'] for r in ok]):.4f}")
print(f"Simulated val: LSTM 0.8214 | Transformer 0.7086")

In [ ]:
import json, pickle

missing = []

def grab(name, fn):
    try:
        return fn()
    except Exception:
        missing.append(name)
        return None

export = {
    'baselines': grab('baselines', lambda: BASELINES),
    'multiseed': grab('multiseed', lambda: {
        k: [{kk: vv for kk, vv in r.items() if kk != 'history'} for r in v]
        for k, v in results.items()}),
    'histories': grab('histories', lambda: {
        k: [r['history'] for r in v] for k, v in results.items()}),
    'horizon_truncated': grab('horizon_truncated', lambda: {
        'horizons_ms': [h / FS * 1000 for h in HORIZONS],
        'lstm': h_lstm, 'transformer': h_trans}),
    'horizon_trained': grab('horizon_trained',
                            lambda: {str(k): v for k, v in horizon_curve.items()}),
    'merger_mode': grab('merger_mode', lambda: {
        'overlap': float(ov_m.mean()), 'std': float(ov_m.std())}),
    'ablation_detector': grab('ablation_detector', lambda: {
        'lstm_h1': float(ov_lstm_h1), 'lstm_dual': float(ov_lstm_dual),
        'trans_h1': float(ov_trans_h1), 'trans_dual': float(ov_trans_dual)}),
    'alpha_sweep': grab('alpha_sweep', lambda: {
        'alphas': list(ALPHA_VALUES), 'lstm': alpha_results['lstm'],
        'transformer': alpha_results['transformer']}),
    'phase': grab('phase', lambda: {
        'lstm_mean': float(np.mean(phase_err_lstm)),
        'lstm_std': float(np.std(phase_err_lstm)),
        'trans_mean': float(np.mean(phase_err_trans)),
        'trans_std': float(np.std(phase_err_trans))}),
    'controls': grab('controls', lambda: control_results),
    'real_events': grab('real_events', lambda: real_results),
    'latency': grab('latency', lambda: {
        'lstm': [lat_lstm_mean, lat_lstm_std],
        'transformer': [lat_trans_mean, lat_trans_std]}),
    'eval': grab('eval', lambda: {
        n: {k: (float(v) if isinstance(v, (int, float)) else None)
            for k, v in r.items()
            if k not in ('preds', 'overlaps', 'phase_err', 'amp_ratio', 'mses')}
        for n, r in [('lstm_val', results_lstm_id), ('trans_val', results_trans_id)]}),
    'config': grab('config', lambda: {
        'context_len': CONTEXT_LEN, 'forecast_len': FORECAST_LEN,
        'fs': FS, 'patch': PATCH, 'bank_mode': BANK_MODE,
        'window_mode': WINDOW_MODE, 'snr_range': list(SNR_RANGE),
        'lstm_params': lstm_model.count_params(),
        'transformer_params': transformer_model.count_params(),
        'lstm_kw': LSTM_KW, 'tf_kw': TF_KW, 'seeds': SEEDS,
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'warmup_epochs': 10, 'es_patience': 15, 'rlrop_patience': 8,
        'tf_version': tf.__version__}),
}

with open('RESULTS_EXPORT.json', 'w') as f:
    json.dump(export, f, indent=2, default=str)

arrays = {}
for nm, fn in [('X_val_sample', lambda: X_val[:200]),
               ('Y_val_sample', lambda: Y_val[:200]),
               ('pred_lstm', lambda: results_lstm_id['preds'][:200]),
               ('pred_trans', lambda: results_trans_id['preds'][:200]),
               ('ov_lstm', lambda: results_lstm_id['overlaps']),
               ('ov_trans', lambda: results_trans_id['overlaps']),
               ('ph_lstm', lambda: results_lstm_id['phase_err']),
               ('ph_trans', lambda: results_trans_id['phase_err']),
               ('snr_val', lambda: snr_val),
               ('params_val', lambda: bank['params_val'])]:
    try:
        arrays[nm] = fn()
    except Exception:
        missing.append('array:' + nm)

np.savez_compressed('arrays_for_figures.npz', **arrays)

print('Saved RESULTS_EXPORT.json and arrays_for_figures.npz')
print('Arrays saved:', list(arrays.keys()))
if missing:
    print('MISSING (cells not run in this kernel):', missing)
print([f for f in os.listdir('.') if f.endswith(('.png', '.csv', '.json', '.pkl', '.npz'))])

In [ ]:
import shutil, time

ref = os.path.getmtime('multiseed_results_final.pkl')
os.makedirs('archive_v1', exist_ok=True)
keep = {'bank_jitter.npz', 'bank_aligned.npz', 'multiseed_results.pkl',
        'multiseed_results_warmup.pkl', 'multiseed_results_final.pkl',
        'RESULTS_EXPORT.json', 'arrays_for_figures.npz', 'training_log.txt'}

moved = []
for f in sorted(os.listdir('.')):
    if not f.endswith(('.png', '.csv', '.pkl')) or f in keep:
        continue
    if os.path.getmtime(f) < ref:
        shutil.move(f, os.path.join('archive_v1', f))
        moved.append(f)

print('Moved to archive_v1:')
for f in moved:
    print('  ', f)
print('\nCurrent figures remaining:')
for f in sorted(os.listdir('.')):
    if f.endswith(('.png', '.csv')):
        print('  ', f, time.strftime('%H:%M', time.localtime(os.path.getmtime(f))))

In [ ]:
import os, pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

C_LSTM, C_TF, C_REF = '#1f77b4', '#d95f02', '#6e6e6e'
FS = 1024
OUT = 'figures_paper'
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.6,
    'lines.linewidth': 2.0, 'lines.markersize': 7,
    'legend.frameon': False, 'figure.dpi': 110,
})

def save(fig, name):
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print('wrote', path)

def load_pkl(p):
    with open(p, 'rb') as f:
        return pickle.load(f)


def fig_parameter_space(bank_file='bank_jitter.npz'):
    d = np.load(bank_file)
    keys = list(d['param_keys'])
    im1, im2 = keys.index('m1'), keys.index('m2')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    sets = [('train', 'Train', C_LSTM, 0.25, 6),
            ('ood_q', 'OOD mass ratio', C_TF, 0.7, 10),
            ('ood_spin', 'OOD spin', '#117733', 0.7, 10),
            ('ood_mass', 'OOD total mass', '#882255', 0.7, 10)]
    for k, lab, c, a, sz in sets:
        p = d[f'params_{k}']
        m1, m2 = p[:, im1], p[:, im2]
        axes[0].scatter(m1 + m2, m1 / m2, s=sz, c=c, alpha=a, label=lab,
                        edgecolors='none')
    axes[0].set_xlabel(r'Total mass $M_{\rm tot}$ ($M_\odot$)')
    axes[0].set_ylabel(r'Mass ratio $q = m_1/m_2$')
    axes[0].set_title('Parameter coverage')
    axes[0].legend(loc='upper right', fontsize=9)

    isz = keys.index('s1z')
    for k, lab, c, a, sz in sets:
        p = d[f'params_{k}']
        axes[1].scatter(p[:, im1] + p[:, im2], p[:, isz], s=sz, c=c, alpha=a,
                        edgecolors='none')
    axes[1].set_xlabel(r'Total mass $M_{\rm tot}$ ($M_\odot$)')
    axes[1].set_ylabel(r'Aligned spin $\chi_1$')
    axes[1].set_title('Spin coverage')
    fig.tight_layout()
    save(fig, 'fig_parameter_space.png')


def fig_training_curves(final='multiseed_results_final.pkl'):
    r = load_pkl(final)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for key, c, lab in [('lstm', C_LSTM, 'LSTM'), ('transformer', C_TF, 'Transformer')]:
        for metric, ax, ylab in [('val_loss', axes[0], 'Validation loss'),
                                 ('val_overlap_tf', axes[1], 'Validation overlap')]:
            series = []
            for rec in r[key]:
                h = rec['history']
                k = metric if metric in h else ('val_overlap_metric'
                                                if 'val_overlap_metric' in h else None)
                if k is None:
                    continue
                series.append(np.asarray(h[k], dtype=float))
            if not series:
                continue
            n = min(len(s) for s in series)
            M = np.stack([s[:n] for s in series])
            ep = np.arange(1, n + 1)
            ax.plot(ep, M.mean(0), color=c, label=lab)
            ax.fill_between(ep, M.min(0), M.max(0), color=c, alpha=0.18, linewidth=0)
            ax.set_xlabel('Epoch'); ax.set_ylabel(ylab)
    axes[0].set_yscale('log')
    axes[0].set_title('Training convergence')
    axes[1].set_title('Validation overlap')
    axes[1].axhline(0.3638, color=C_REF, ls='--', lw=1.5, label='AR-64 baseline')
    axes[0].legend(); axes[1].legend(loc='lower right')
    fig.tight_layout()
    save(fig, 'fig_training_curves.png')


def fig_seed_distribution():
    protos = [('multiseed_results.pkl', 'No warmup\npatience 7'),
              ('multiseed_results_warmup.pkl', 'Patience 15\n(no warmup)'),
              ('multiseed_results_final.pkl', 'Warmup 10\npatience 15')]
    have = [(f, lab) for f, lab in protos if os.path.exists(f)]
    if not have:
        print('skip seed distribution: no pkl files'); return
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    width = 0.16
    for i, (f, lab) in enumerate(have):
        r = load_pkl(f)
        for j, (key, c) in enumerate([('lstm', C_LSTM), ('transformer', C_TF)]):
            vals = [rec['val_overlap'] for rec in r.get(key, [])]
            if not vals:
                continue
            x = np.full(len(vals), i + (j - 0.5) * 0.32)
            x = x + np.linspace(-width / 2, width / 2, len(vals))
            ax.scatter(x, vals, s=55, c=c, alpha=0.9, edgecolors='white',
                       linewidths=0.8, zorder=3,
                       label=('LSTM' if j == 0 else 'Transformer') if i == 0 else None)
            ax.plot([i + (j - 0.5) * 0.32 - 0.11, i + (j - 0.5) * 0.32 + 0.11],
                    [np.mean(vals)] * 2, color=c, lw=2.5, zorder=2)
    ax.axhline(0.3638, color=C_REF, ls='--', lw=1.5, label='AR-64 baseline')
    ax.set_xticks(range(len(have)))
    ax.set_xticklabels([lab for _, lab in have])
    ax.set_ylabel('Validation overlap')
    ax.set_title('Per-seed outcomes under three training protocols')
    ax.legend(loc='lower right', fontsize=9)
    fig.tight_layout()
    save(fig, 'fig_seed_distribution.png')


def _binned(x, y, nbins=10):
    edges = np.linspace(np.nanmin(x), np.nanmax(x), nbins + 1)
    ctr, mu, lo, hi = [], [], [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (x >= a) & (x < b)
        if m.sum() < 5:
            continue
        ctr.append(0.5 * (a + b)); mu.append(np.mean(y[m]))
        lo.append(np.percentile(y[m], 25)); hi.append(np.percentile(y[m], 75))
    return np.array(ctr), np.array(mu), np.array(lo), np.array(hi)


def fig_overlap_vs(arrays='arrays_for_figures.npz', bank_file='bank_jitter.npz'):
    a = np.load(arrays)
    need = ['ov_lstm', 'ov_trans', 'snr_val']
    if not all(k in a.files for k in need):
        print('skip overlap-vs: missing arrays'); return
    ovl, ovt, snr = a['ov_lstm'], a['ov_trans'], a['snr_val']
    n = min(len(ovl), len(ovt), len(snr))
    ovl, ovt, snr = ovl[:n], ovt[:n], snr[:n]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for y, c, lab in [(ovl, C_LSTM, 'LSTM'), (ovt, C_TF, 'Transformer')]:
        cx, mu, lo, hi = _binned(snr, y)
        axes[0].plot(cx, mu, 'o-', color=c, label=lab)
        axes[0].fill_between(cx, lo, hi, color=c, alpha=0.16, linewidth=0)
    axes[0].axhline(0.3638, color=C_REF, ls='--', lw=1.5, label='AR-64 baseline')
    axes[0].set_xlabel('Injected network SNR'); axes[0].set_ylabel('Overlap')
    axes[0].set_title('Fidelity vs signal strength')
    axes[0].legend(loc='lower right', fontsize=9)

    pv = a['params_val'] if 'params_val' in a.files else None
    if pv is not None:
        d = np.load(bank_file); keys = list(d['param_keys'])
        mt = pv[:n, keys.index('m1')] + pv[:n, keys.index('m2')]
        for y, c, lab in [(ovl, C_LSTM, 'LSTM'), (ovt, C_TF, 'Transformer')]:
            cx, mu, lo, hi = _binned(mt, y)
            axes[1].plot(cx, mu, 'o-', color=c, label=lab)
            axes[1].fill_between(cx, lo, hi, color=c, alpha=0.16, linewidth=0)
        axes[1].axhline(0.3638, color=C_REF, ls='--', lw=1.5)
        axes[1].set_xlabel(r'Total mass $M_{\rm tot}$ ($M_\odot$)')
        axes[1].set_ylabel('Overlap')
        axes[1].set_title('Fidelity vs total mass (in distribution)')
        axes[1].legend(loc='lower left', fontsize=9)
    fig.tight_layout()
    save(fig, 'fig_overlap_vs.png')


if __name__ == '__main__':
    for fn in [fig_parameter_space, fig_training_curves,
               fig_seed_distribution, fig_overlap_vs]:
        try:
            fn()
        except Exception as e:
            print(f'SKIP {fn.__name__}: {type(e).__name__}: {e}')
    print('\nDone. Figures in', OUT)
